> **PRE-RUN COPY.** The outputs below were produced on 2026-10-13 on the team's own laptop (Windows, CPU only, 12 threads) by running every cell of this notebook. They are saved so the notebook can be shown without running it. To regenerate them on Colab or Kaggle, use **Run all**; neural-network numbers may differ by about 0.01 m.

# TGT-M: the implementation, shown live (Review 2 demo, about 5 minutes)
**Team 7 · 21CSC314P Big Data Essentials.** Title: *TGT-M: A Temporal Gated Transformer Model for Seasonal Groundwater Stress Assessment and Cross-Region Generalization Across India Using Multimodal Data*.

This notebook **shows the real code and recomputes the published numbers**; it does not retrain the whole study (that takes hours and is not needed for the review).

| Section | What it shows |
|---|---|
| 1 | Finds the project files (zip upload, Kaggle dataset, or GitHub) |
| 2 | The data, and the **anomaly / stress label computed by hand** and matched with the pipeline (baseline uses only earlier readings) |
| 3 | What the model receives: a **13-step sequence tensor** whose current reading is hidden; no location columns |
| 4 | The **source code** of the TGT-M building blocks |
| 5 | **Build TGT-M**, count parameters, run one forward pass and see every shape |
| 6 | A **mini training run** (6,000 rows, about a minute) to show the training loop learns |
| 7 | The **report's test errors recomputed** from saved predictions and checked against the stored tables |
| 8 | The two **pre-registered tests** (tables) and the verdicts |


## 1 · Find the project files
Colab: upload `bde_bundle_core.zip` in the Files panel. Kaggle: add it as a dataset (Add Data). The cell finds it automatically.

In [1]:
import os, sys, glob, zipfile, subprocess, time, shutil
from pathlib import Path
BUNDLE_ZIP = None      # e.g. "/content/bde_bundle_core.zip" if auto-detect fails
GITHUB_URL = "https://github.com/dhruvvvgg/bde-team7.git"   # only works after the project was pushed

ON_KAGGLE = Path("/kaggle").exists()
ON_COLAB = (not ON_KAGGLE) and Path("/content").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content") if ON_COLAB else Path.cwd()

def looks_ok(p):
    p = Path(p); return (p / "modeling" / "data.py").exists() and (p / "processed" / "gw_features_by_state").exists()

def writable(c):
    """Kaggle mounts datasets read-only; copy the project to the writable working folder so quick/full mode can write results."""
    c = Path(c)
    if str(c).startswith("/kaggle/input"):
        dest = WORK / "bde-team7"
        if not looks_ok(dest):
            print("copying the project from the read-only dataset to", dest, "...")
            shutil.copytree(c, dest, dirs_exist_ok=True)
        return dest
    return c

def find_root():
    cands = [os.environ.get("BDE_ROOT"), Path.cwd(), Path.cwd().parent, WORK / "bde-team7"]
    cands += [Path(p).parent.parent for p in glob.glob("/kaggle/input/**/modeling/data.py", recursive=True)]
    for c in cands:
        if c and looks_ok(c): return writable(c)
    zips = ([BUNDLE_ZIP] if BUNDLE_ZIP else []) + glob.glob("/kaggle/input/**/bde_bundle_core*.zip", recursive=True) + glob.glob("/content/**/bde_bundle_core*.zip", recursive=True) + glob.glob(str(Path.cwd() / "bde_bundle_core*.zip"))
    for z in zips:
        if z and Path(z).exists():
            print("unzipping", Path(z).name); zipfile.ZipFile(z).extractall(WORK)
            for c in (WORK / "bde-team7", WORK):
                if looks_ok(c): return c
    dest = WORK / "bde-team7"
    if not dest.exists(): subprocess.run(["git", "clone", "-q", "--depth", "1", GITHUB_URL, str(dest)])
    if looks_ok(dest): return dest
    raise SystemExit("Project files not found: upload bde_bundle_core.zip (Colab: Files panel; Kaggle: Add Data) and rerun.")

ROOT = find_root(); os.environ["BDE_ROOT"] = str(ROOT); os.chdir(ROOT)
print("platform:", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "local machine", "| project folder:", ROOT.name, "| cpus:", os.cpu_count())

platform: local machine | project folder: bde-team7 | cpus: 12


In [2]:
import importlib
miss = [p for m, p in {"xgboost": "xgboost", "pyarrow": "pyarrow", "sklearn": "scikit-learn", "scipy": "scipy"}.items() if importlib.util.find_spec(m) is None]
if miss: subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + miss)
import torch                                   # import torch BEFORE pandas (matters on Windows; harmless elsewhere)
import numpy as np, pandas as pd
sys.path.insert(0, str(ROOT / "modeling")); sys.path.insert(0, str(ROOT / "src"))
print("torch", torch.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

torch 2.14.0+cpu | pandas 3.0.2 | numpy 2.5.3


## 2 · The data and the label, computed by hand
One real well: **Cherfully, Thane (Maharashtra)**, a dug well. We recompute the May-2006 baseline, anomaly (z) and stress label ourselves and compare with the stored pipeline values.
Rule: *baseline = mean and SD of the **earlier** May readings (at least 5)*; *z = (reading − mean) / SD*; *High Stress if z ≥ 2*.

In [3]:
m = pd.read_parquet(ROOT / "processed" / "gw_features_by_state",
                    columns=["well_id", "station_name", "district", "season", "date", "period_label", "gwl_m_bgl", "baseline_mean_m", "baseline_std_m", "anomaly_z", "stress_category"])
w = m[(m.well_id == "Wa62f7df166") & (m.season == "pre_monsoon")].sort_values("date").reset_index(drop=True)
print(w.station_name.iloc[0], "|", w.district.iloc[0])
i = w.index[w.period_label == "May-06"][0]
earlier = w.loc[:i - 1, "gwl_m_bgl"].dropna()
mean, sd = earlier.mean(), earlier.std(ddof=1)          # sample SD, only EARLIER readings
z = (w.gwl_m_bgl[i] - mean) / max(sd, 0.10)
print("earlier May readings used:", earlier.round(2).tolist())
print(f"computed here : mean {mean:.3f}  SD {sd:.3f}  z {z:.3f}  -> {'High Stress' if z >= 2 else 'not High Stress'}")
print(f"pipeline value: mean {w.baseline_mean_m[i]:.3f}  SD {w.baseline_std_m[i]:.3f}  z {w.anomaly_z[i]:.3f}  -> {w.stress_category[i]}")
display(w[["period_label", "gwl_m_bgl", "baseline_mean_m", "baseline_std_m", "anomaly_z", "stress_category"]].round(2).head(14))

Cherfully | Thane
earlier May readings used: [1.25, 1.54, 1.7, 1.65, 2.28]
computed here : mean 1.684  SD 0.376  z 2.037  -> High Stress
pipeline value: mean 1.684  SD 0.376  z 2.037  -> High Stress


,period_label,gwl_m_bgl,baseline_mean_m,baseline_std_m,anomaly_z,stress_category
0,May-00,1.25,NaN,NaN,NaN,Insufficient history
1,May-01,1.54,NaN,NaN,NaN,Insufficient history
2,May-02,NaN,NaN,NaN,NaN,No reading
3,May-03,1.70,NaN,NaN,NaN,Insufficient history
4,May-04,1.65,NaN,NaN,NaN,Insufficient history
5,May-05,2.28,NaN,NaN,NaN,Insufficient history
6,May-06,2.45,1.68,0.38,2.04,High Stress
7,May-07,2.20,1.81,0.46,0.85,Normal
8,May-08,0.97,1.87,0.44,-2.02,Normal
9,May-09,2.80,1.76,0.52,2.01,High Stress


In [4]:
print("rows:", len(m), "| wells:", m.well_id.nunique(), "| readings:", int(m.gwl_m_bgl.notna().sum()), "| missing: %.2f%%" % (100 * m.gwl_m_bgl.isna().mean()))
print(m.stress_category.astype(str).value_counts().to_string())

rows: 253828 | wells: 2759 | readings: 219258 | missing: 13.62%


stress_category
Normal                  135523
Insufficient history     52952
No reading               34570
Watch                    12661
High Stress              10609
Moderate Stress           7513


## 3 · What the model receives
`modeling/data.py` builds, for every eligible row, a **tabular vector** and a **sequence tensor**: 13 steps (12 past rounds + the current round) × 10 channels. The current round's **reading is always hidden** (mask 0); only its season and rainfall up to the previous month are visible. Depth lags are expressed relative to the well's own baseline, and latitude, longitude, state and district are **never** inputs.

In [5]:
import data as D
b = D.build(); meta = b["meta"]
print("eligible rows:", len(meta), "| tabular inputs A / B:", len(b["names"]) - int(b["is_rain"].sum()), "/", len(b["names"]), "| sequence tensor:", b["seq"].shape)
print("splits:", meta.split_chrono.value_counts().to_dict())
banned = {"lat", "lon", "state", "district", "year"}
print("forbidden location/year columns among the inputs:", sorted(banned & set(b["names"])) or "none")
k = int(np.flatnonzero((meta.well_id == "Wa62f7df166") & (meta.period_label == "May-06"))[0])
ch = ["reading(rel)", "mask", "Jan", "May", "Aug", "Nov", "valid", "log rain1m", "log rain3m", "spare"]
print("\nsample: well Wa62f7df166, round May-06 (last row = the round to predict, reading hidden)")
display(pd.DataFrame(b["seq"][k], columns=ch, index=[f"t-{12 - j}" if j < 12 else "t (hidden)" for j in range(13)]).round(2))

eligible rows: 160580 | tabular inputs A / B: 42 / 66 | sequence tensor: (160580, 13, 10)
splits: {'train': 104437, 'test': 40630, 'val': 15513}
forbidden location/year columns among the inputs: none

sample: well Wa62f7df166, round May-06 (last row = the round to predict, reading hidden)


,reading(rel),mask,Jan,May,Aug,Nov,valid,log rain1m,log rain3m,spare
t-12,0.04,1.0,0.0,1.0,0.0,0.0,1.0,0.00,0.24,0.0
t-11,-3.23,1.0,0.0,0.0,1.0,0.0,1.0,6.93,7.31,0.0
t-10,1.16,1.0,0.0,0.0,0.0,1.0,1.0,3.94,6.62,0.0
t-9,-0.76,1.0,1.0,0.0,0.0,0.0,1.0,0.53,3.98,0.0
t-8,-0.09,1.0,0.0,1.0,0.0,0.0,1.0,0.00,0.18,0.0
t-7,0.00,0.0,0.0,0.0,1.0,0.0,1.0,7.20,7.45,0.0
t-6,0.00,0.0,0.0,0.0,0.0,1.0,1.0,3.74,7.34,0.0
t-5,-1.23,1.0,1.0,0.0,0.0,0.0,1.0,0.30,3.98,0.0
t-4,1.58,1.0,0.0,1.0,0.0,0.0,1.0,0.00,0.18,0.0
t-3,-2.62,1.0,0.0,0.0,1.0,0.0,1.0,7.13,7.54,0.0


## 4 · The TGT-M source code
The building blocks as implemented in `modeling/nets.py` (GRU-D encoder, variable-selection network, gated residual network, and the full model).

In [6]:
import inspect, nets as N
for obj in (N.GRUD, N.GRN, N.VSN2):
    print(inspect.getsource(obj)); print("-" * 90)

class GRUD(nn.Module):
    """GRU-D encoder. Channel 0 = relative reading, 1 = mask; other channels are always observed."""

    def __init__(self, n_ch, d):
        super().__init__()
        self.d = d
        self.gx = nn.Linear(1, 1)
        self.gh = nn.Linear(1, d)
        self.cell = nn.GRUCell(n_ch, d)

    def forward(self, seq):
        B, T, _ = seq.shape
        h = seq.new_zeros(B, self.d)
        x_last = seq.new_zeros(B, 1)
        delta = seq.new_zeros(B, 1)
        prev_m = seq.new_ones(B, 1)
        out = []
        for t in range(T):
            x, m = seq[:, t, 0:1], seq[:, t, 1:2]
            delta = torch.where(prev_m > 0, torch.ones_like(delta), delta + 1) if t else delta
            dn = delta / 12.0
            gamma_x = torch.exp(-F.relu(self.gx(dn)))
            x_hat = m * x + (1 - m) * gamma_x * x_last         # decay toward the well's baseline (0)
            gamma_h = torch.exp(-F.relu(self.gh(dn)))
            h = self.cell(torch.cat([x_hat, m, seq[:, t,

In [7]:
print(inspect.getsource(N.TGTM2))

class TGTM2(TGTM):
    """TGT-M v2: per-variable VSN non-linearity and a skip of the selected static context to the output."""

    def __init__(self, n_static, hidden=48, drop=0.1, n_ch=10, heads=2, rain_idx=None, **_):
        super().__init__(n_static, hidden, drop, n_ch, heads, rain_idx)
        self.vsn = VSN2(n_static, hidden, drop)
        self.mix = nn.Linear(2 * hidden, hidden)

    def forward(self, seq, st, return_all=False):
        H = self.enc(seq)
        c, w = self.vsn(st)
        self.last_w = w.detach()
        E = self.enrich(H, c.unsqueeze(1).expand_as(H))
        pad = seq[:, :, 6] < 0.5
        a, aw = self.attn(E[:, -1:], E, E, key_padding_mask=pad, need_weights=True)
        z = self.norm(E[:, -1] + self.gate(a.squeeze(1)))
        z = self.norm(z + F.elu(self.mix(torch.cat([z, c], -1))))
        o = self.out(self.pos(z))
        self.last_attn = aw.detach()
        return (o[:, 0], o[:, 1:]) if return_all else o[:, 0]



## 5 · Build TGT-M and run one forward pass
`TGTM2` (exposed as `"TGT-M"`): **GRU-D** encoder (mask + time decay) → **variable-selection network** → **gated residual** enrichment → **attention** with the current round as query → heads: point, q10, q50, q90.

In [8]:
import lib as Lb
cfg = Lb.GRIDS["TGT-M"][2]                     # configuration chosen on validation for Model B
Bq = Lb.Bundle(b, True, (meta.split_chrono == "train").to_numpy())
net = N.NETS["TGT-M"](n_static=Bq.X.shape[1], rain_idx=np.flatnonzero(Bq.is_rain), **cfg["arch"])
print("config:", cfg)
print("parameters:", f"{sum(p.numel() for p in net.parameters()):,}")
net.eval()
seq, st = torch.from_numpy(Bq.seq[:8]), torch.from_numpy(Bq.X[:8])
with torch.no_grad():
    Hs = net.enc(seq); c, wts = net.vsn(st); point, q = net(seq, st, return_all=True)
print("input sequence       :", tuple(seq.shape), "   static vector:", tuple(st.shape))
print("GRU-D states         :", tuple(Hs.shape), "   (one state per step)")
print("VSN context / weights:", tuple(c.shape), "/", tuple(wts.shape), "(weights sum to", round(float(wts.sum(1).mean()), 3), ")")
print("attention weights    :", tuple(net.last_attn.shape), "(the current round attends over the 13 steps)")
print("point / quantiles    :", tuple(point.shape), "/", tuple(q.shape), " (untrained network, random output)")

config: {'arch': {'hidden': 48, 'drop': 0.3, 'heads': 2}, 'lr': 0.0005, 'wd': 0.01}
parameters: 75,646


input sequence       : (8, 13, 10)    static vector: (8, 88)
GRU-D states         : (8, 13, 48)    (one state per step)
VSN context / weights: (8, 48) / (8, 88) (weights sum to 1.0 )
attention weights    : (8, 1, 13) (the current round attends over the 13 steps)
point / quantiles    : (8,) / (8, 3)  (untrained network, random output)


## 6 · Mini training run (6,000 rows, about a minute)
The same training function as the study (`lib.fit_deep`: AdamW, Huber + pinball loss, early stopping), on a small random sample so it finishes quickly. The published numbers come from the full runs.

In [9]:
import warnings; warnings.filterwarnings("ignore")
tr = (meta.split_chrono == "train").to_numpy(); va = (meta.split_chrono == "val").to_numpy()
rng = np.random.default_rng(0)
tr_s = np.zeros(len(meta), bool); tr_s[rng.choice(np.flatnonzero(tr), 6000, replace=False)] = True
va_s = np.zeros(len(meta), bool); va_s[rng.choice(np.flatnonzero(va), 2000, replace=False)] = True
y = meta["anomaly_z_capped"].to_numpy()
torch.manual_seed(0)
t0 = time.time()
net2, best, ep = Lb.fit_deep("TGT-M", Bq, y, tr_s, va_s, cfg, 0, epochs=6, patience=3, verbose=True)
print(f"trained on 6,000 rows in {time.time() - t0:.0f}s; best validation |z| MAE {best:.3f}  (always predicting z = 0 gives {np.abs(y[va_s]).mean():.3f})")

  ep0 val |z| MAE 0.9596


  ep1 val |z| MAE 0.9428


  ep2 val |z| MAE 0.9102


  ep3 val |z| MAE 0.8999


  ep4 val |z| MAE 0.8874


  ep5 val |z| MAE 0.8604


trained on 6,000 rows in 32s; best validation |z| MAE 0.860  (always predicting z = 0 gives 0.965)


## 7 · The report's numbers, recomputed from saved predictions
`results/powerbi/test_predictions.csv` holds each model's predicted depth for every test row (seed 0). We recompute the depth MAE ourselves and compare it with the stored per-seed table (the same numbers as in the report).

In [10]:
tp = pd.read_csv(ROOT / "results" / "powerbi" / "test_predictions.csv")
seed0 = pd.read_csv(ROOT / "results" / "tables" / "chrono_per_seed.csv"); seed0 = seed0[seed0.seed == 0].set_index(["model", "variant"])
rows = []
for col in [c for c in tp.columns if c.endswith("_depth_pred")]:
    mod, var = col.split("_")[0], col.split("_")[1]
    mine = float(np.abs(tp[col] - tp.gwl_m_bgl).mean())
    stored = float(seed0.loc[(mod, var), "depth_MAE"]) if (mod, var) in seed0.index else np.nan
    rows.append([mod, var, round(mine, 3), round(stored, 3), "match" if abs(mine - stored) < 1e-3 else "CHECK"])
out = pd.DataFrame(rows, columns=["model", "variant", "recomputed depth MAE (m)", "stored in report tables (m)", "check"]).sort_values(["variant", "recomputed depth MAE (m)"])
print("test rows:", len(tp), "| seasonal-baseline depth MAE for reference:", round(float(seed0.loc[("Seasonal baseline", "B"), "depth_MAE"]), 3), "m")
display(out)

test rows: 40630 | seasonal-baseline depth MAE for reference: 1.548 m


,model,variant,recomputed depth MAE (m),stored in report tables (m),check
2,RF,A,1.251,1.251,match
0,XGBoost,A,1.260,1.260,match
4,LSTM,A,1.265,1.265,match
6,CNN1D,A,1.279,1.279,match
8,TGT-M,A,1.329,1.329,match
7,CNN1D,B,1.093,1.093,match
5,LSTM,B,1.102,1.102,match
1,XGBoost,B,1.122,1.122,match
9,TGT-M,B,1.137,1.137,match
3,RF,B,1.150,1.150,match


In [11]:
s = pd.read_csv(ROOT / "results" / "tables" / "chrono_summary.csv")
print("Mean over 3 seeds (the numbers quoted in the report), Model B = with rainfall:")
display(s[s.variant == "B"][["model", "depth_MAE_mean", "depth_MAE_std", "anom_R2_mean", "AUC_stressed_mean"]].round(3).sort_values("depth_MAE_mean").reset_index(drop=True))
print("TGT-M is 5th of 7 learned models on depth MAE; rainfall (B vs A) improves every model.")

Mean over 3 seeds (the numbers quoted in the report), Model B = with rainfall:


,model,depth_MAE_mean,depth_MAE_std,anom_R2_mean,AUC_stressed_mean
0,LSTM,1.104,0.003,0.297,0.801
1,CNN1D,1.113,0.019,0.293,0.797
2,XGBoost,1.121,0.001,0.277,0.788
3,ANN,1.125,0.017,0.283,0.793
4,TGT-M,1.129,0.008,0.279,0.795
5,RF,1.150,0.000,0.263,0.781
6,SVR,1.180,0.007,0.250,0.781
7,Same season LY,1.507,NaN,-0.175,0.678
8,Seasonal baseline,1.548,NaN,-0.017,0.500
9,Prev. round,2.050,NaN,-1.731,0.680


TGT-M is 5th of 7 learned models on depth MAE; rainfall (B vs A) improves every model.


## 8 · The two pre-registered tests (from the saved result tables)
Hypotheses, metrics and pass criteria were written down in `docs/preregistration_review2.md` before these tests were run.

In [12]:
T = ROOT / "results" / "tables"
print("H1: 80% intervals (lower interval score is better; nominal coverage 0.80)")
display(pd.read_csv(T / "interval_summary.csv").round(3).head(6))
print("H2: anomaly MAE when 0-60% of the recent readings are removed at test time (all models retrained, identical masks)")
display(pd.read_csv(T / "robust_summary.csv").pivot(index="model", columns="p", values="mean").round(4))
print("Verdicts: H1 supported; H2a (smaller loss than LSTM and 1D-CNN) supported; H2b (lowest error at 60% removal) NOT supported.")

H1: 80% intervals (lower interval score is better; nominal coverage 0.80)


,method,coverage,coverage_sd,width,interval_score,interval_score_sd
0,TGT-M native,0.792,0.023,2.319,3.648,0.008
1,TGT-M + CQR,0.779,0.009,2.249,3.647,0.004
2,XGBoost-QR native,0.806,0.000,2.503,3.698,0.001
3,XGBoost-QR + CQR,0.785,0.001,2.401,3.700,0.001
4,SVR point + conformal,0.803,0.002,2.474,3.883,0.004
5,RF point + conformal,0.810,0.001,2.467,3.854,0.001


H2: anomaly MAE when 0-60% of the recent readings are removed at test time (all models retrained, identical masks)


p,0.0,0.2,0.4,0.6
model,,,,
ANN,0.7597,0.7875,0.8169,0.8471
CNN1D,0.7556,0.7830,0.8122,0.8445
LSTM,0.7475,0.7748,0.8048,0.8386
RF,0.7767,0.7937,0.8131,0.8362
SVR,0.7904,0.8179,0.8479,0.8797
TGT-M,0.7627,0.7854,0.8128,0.8450
XGBoost,0.7582,0.7850,0.8142,0.8478


Verdicts: H1 supported; H2a (smaller loss than LSTM and 1D-CNN) supported; H2b (lowest error at 60% removal) NOT supported.


**End of demo.** Honest summary for the panel: TGT-M ranks 5th of 7 learned models on accuracy; its demonstrated strengths are better 80% intervals and a smaller loss than LSTM/CNN when recent readings are missing (pre-registered tests); rainfall improves every model.